# Bonus (optional, advanced) — Sparse autoencoders & emerging signal

This track is **not required** for the core deliverable — it's here if your group finishes
early and wants to see what unsupervised representation learning can find, with *zero*
labels, inside a genomic language model.

## Part 1 — Vanilla autoencoder recap

An autoencoder learns to compress its input through a narrow **bottleneck** and reconstruct
it — the bottleneck forces it to keep only the most useful structure. Let's see this on our
own k-mer vectors first, since it's fast and needs nothing from Evo2.

In [ ]:
import sys
sys.path.append("../src")

import torch
import torch.nn as nn
from data import load_all
from featurize import kmer_matrix

train_df = load_all("../../data/supervised/processed")["train"]
X = torch.tensor(kmer_matrix(train_df["sequence"].tolist(), k=4), dtype=torch.float32)

d_in = X.shape[1]
vanilla_ae = nn.Sequential(
    nn.Linear(d_in, 16), nn.ReLU(),          # <- bottleneck: 256 kmers -> 16 numbers
    nn.Linear(16, d_in),
)
optimizer = torch.optim.Adam(vanilla_ae.parameters(), lr=1e-3)

for epoch in range(20):
    optimizer.zero_grad()
    x_hat = vanilla_ae(X)
    loss = nn.functional.mse_loss(x_hat, X)
    loss.backward()
    optimizer.step()
print(f"final reconstruction loss: {loss.item():.5f}")

This bottleneck is **dense**: every one of the 16 numbers is used for every input, and
they're not individually interpretable — a single "feature" is usually a blend of many
underlying causes (this is called *polysemanticity*).

## Part 2 — What a sparse autoencoder adds

A **sparse** autoencoder (SAE) does the opposite trick: instead of a *narrow* bottleneck, it
uses an **overcomplete** one (more hidden units than inputs), but forces only a handful
(`k`) to be active for any given input. The idea (from recent mechanistic interpretability
research on language models): with enough capacity and the right sparsity, individual
features tend to become **monosemantic** — each one tracks one distinguishable pattern,
which makes them inspectable.

Here we train an SAE on **raw Evo2 activations** — per-nucleotide-position internal states
from the model, extracted with no label information at all — and look for features that
line up with something real, like the reading frame of a codon (period-3 structure).

In [ ]:
from embeddings import load_autoencoder_activations
from models.sae import TopKSparseAutoencoder, train_sae

acts = load_autoencoder_activations("../../data/autoencoder", "train")
print("full activation dump shape:", acts.shape)

# subsample into memory — this is a memmap over a 19GB file, don't load it whole
N_SUBSAMPLE = 20_000
X_acts = torch.tensor(acts[:N_SUBSAMPLE].astype("float32"))
print("training on:", X_acts.shape)

In [ ]:
sae = TopKSparseAutoencoder(d_in=X_acts.shape[1], d_hidden=X_acts.shape[1] * 8, k=32)
sae, history = train_sae(sae, X_acts, epochs=10, batch_size=512)

## Part 3 — Look for emerging structure: periodicity

DNA is read three nucleotides at a time (codons). If the SAE has discovered anything about
reading frame with zero supervision, some features should activate periodically with
period ~3 along a contiguous stretch of positions.

In [ ]:
import numpy as np
from viz import plot_sae_feature_activity

# a contiguous slice of consecutive token positions from the memmap
window = torch.tensor(acts[1000:1000 + 300].astype("float32"))
with torch.no_grad():
    _, features = sae(window)
features = features.numpy()

# rank features by activation variance (a cheap proxy for "does something interesting")
variances = features.var(axis=0)
top_features = np.argsort(variances)[::-1][:5]

plot_sae_feature_activity(features[:, top_features], top_features)

In [ ]:
# quick periodicity check via FFT — look for a peak near frequency 1/3
for feat_idx in top_features:
    signal = features[:, feat_idx] - features[:, feat_idx].mean()
    spectrum = np.abs(np.fft.rfft(signal))
    freqs = np.fft.rfftfreq(len(signal))
    period = 1 / freqs[np.argmax(spectrum[1:]) + 1] if len(spectrum) > 1 else float("nan")
    print(f"feature {feat_idx}: dominant period ~ {period:.2f} positions")

### Discussion

- Did any feature come out with a period close to 3? That would suggest the model — with
  no labels, no task, just "reconstruct my own internal activations" — has represented
  reading frame internally.
- Try correlating top features against the coding/non-coding labels from
  `data/supervised/processed` for the same genome region (you'll need to align token
  positions to genome coordinates) — do any features separate coding from non-coding
  regions on their own?
- This is exploratory, not graded on getting a "real" result — the point is seeing that
  interpretable structure *can* emerge from pure unsupervised training.